# DDP Advanced: Performance Optimization and Debugging

## Overview

This advanced tutorial covers DDP performance optimization, debugging techniques, and production best practices.

### Topics Covered
- Gradient bucket optimization
- Communication profiling
- Debugging distributed training
- Production deployment patterns

### Prerequisites
- Complete 01_ddp_tutorial.ipynb first

## 1. Gradient Bucket Optimization

DDP groups gradients into buckets for efficient AllReduce communication.

### Bucket Size Trade-offs

```
Small Buckets (1-5 MB):
├── Better overlap with backward computation
├── Higher per-bucket latency overhead
└── Best for: High-latency networks, small models

Large Buckets (25-50 MB):
├── Better bandwidth utilization
├── Less overlap opportunity
└── Best for: High-bandwidth networks, large models
```

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def create_optimized_ddp(model, rank, bucket_cap_mb=25):
    """Create DDP with optimized bucket settings."""
    model = model.to(rank)
    
    ddp_model = DDP(
        model,
        device_ids=[rank],
        bucket_cap_mb=bucket_cap_mb,
        gradient_as_bucket_view=True,  # Memory optimization
        static_graph=True,  # Enable for static models
    )
    return ddp_model

# Benchmark different bucket sizes
def benchmark_bucket_sizes(model_fn, sizes=[5, 15, 25, 50]):
    results = {}
    for size in sizes:
        # Measure throughput with this bucket size
        results[size] = f"bucket_cap_mb={size}"
    return results

## 2. Communication Profiling

In [ ]:
import time

class DDPProfiler:
    """Profile DDP communication overhead."""
    
    def __init__(self):
        self.forward_times = []
        self.backward_times = []
        self.comm_times = []
    
    def profile_step(self, model, data, target, criterion):
        """Profile a single training step."""
        # Forward
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        output = model(data)
        loss = criterion(output, target)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        self.forward_times.append(t1 - t0)
        
        # Backward (includes communication)
        t2 = time.perf_counter()
        loss.backward()
        torch.cuda.synchronize()
        t3 = time.perf_counter()
        self.backward_times.append(t3 - t2)
        
        return loss.item()
    
    def report(self):
        """Print profiling report."""
        print(f"Forward:  {sum(self.forward_times)/len(self.forward_times)*1000:.2f} ms")
        print(f"Backward: {sum(self.backward_times)/len(self.backward_times)*1000:.2f} ms")

## 3. Debugging Distributed Training

### Common Issues and Solutions

| Issue | Symptom | Solution |
|-------|---------|----------|
| Gradient mismatch | Loss diverges | Check random seeds, use DistributedSampler |
| Deadlock | Training hangs | Ensure all ranks execute same ops |
| OOM on some ranks | Crash | Balance batch sizes across GPUs |
| Slow communication | Low GPU util | Check network, increase bucket size |

In [ ]:
def debug_gradient_sync(model):
    """Check if gradients are synchronized across ranks."""
    if not dist.is_initialized():
        print("Distributed not initialized")
        return
    
    rank = dist.get_rank()
    
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_sum = param.grad.sum().item()
            
            # Gather all gradient sums
            all_sums = [torch.zeros(1) for _ in range(dist.get_world_size())]
            dist.all_gather(all_sums, torch.tensor([grad_sum]))
            
            if rank == 0:
                sums = [s.item() for s in all_sums]
                if max(sums) - min(sums) > 1e-5:
                    print(f"WARNING: {name} gradients not synced: {sums}")

## 4. Production Best Practices

### Fault Tolerance

In [ ]:
class FaultTolerantTrainer:
    """DDP trainer with automatic checkpoint recovery."""
    
    def __init__(self, model, checkpoint_dir, save_interval=1000):
        self.model = model
        self.checkpoint_dir = checkpoint_dir
        self.save_interval = save_interval
        self.step = 0
    
    def save_checkpoint(self, optimizer):
        """Save checkpoint (rank 0 only)."""
        if dist.get_rank() == 0:
            torch.save({
                'step': self.step,
                'model': self.model.module.state_dict(),
                'optimizer': optimizer.state_dict(),
            }, f"{self.checkpoint_dir}/ckpt_{self.step}.pt")
        dist.barrier()
    
    def load_latest_checkpoint(self, optimizer):
        """Load most recent checkpoint."""
        import glob
        ckpts = glob.glob(f"{self.checkpoint_dir}/ckpt_*.pt")
        if ckpts:
            latest = max(ckpts, key=lambda x: int(x.split('_')[-1].split('.')[0]))
            ckpt = torch.load(latest, map_location='cpu')
            self.model.module.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            self.step = ckpt['step']
            print(f"Resumed from step {self.step}")

## 5. Summary

### Key Optimization Techniques

1. **Bucket tuning**: Match bucket size to network characteristics
2. **Static graph**: Enable for models with fixed computation graph
3. **Gradient compression**: Use for bandwidth-limited scenarios
4. **Overlap**: Maximize compute-communication overlap

### Debugging Checklist

- [ ] All ranks use same random seed for model init
- [ ] DistributedSampler used for data loading
- [ ] sampler.set_epoch() called each epoch
- [ ] No rank-specific control flow in forward/backward
- [ ] Gradients synchronized (use debug_gradient_sync)